# COMPX523-26A Assignment 1 - Experiments
**Student ID:** 1689664

This notebook installs CapyMOA, defines the KNN-cw and KNN-asw classifiers, runs all 24 experiments, and generates the CSV result file.

## 1. Setup & Installation

In [ ]:
# Install Java and CapyMOA (Colab only)
# Fix numpy first to avoid dependency conflicts, then install capymoa
!apt-get install -y default-jdk -qq
!pip install "numpy<2.1" -q --upgrade
!pip install capymoa -q

# IMPORTANT: After this cell runs, go to Runtime > Restart Session
# Then run all cells from Cell 2 onwards (skip this installation cell)
print("Installation done. Please restart the runtime now (Runtime > Restart Session)")
print("Then run all remaining cells.")

In [ ]:
import os
# Set JAVA_HOME for Colab
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

from capymoa.base import Classifier
from capymoa.evaluation import prequential_evaluation
from capymoa.datasets import Electricity, RBFm_100k
from moa.classifiers.lazy.neighboursearch import KDTree, LinearNNSearch
from com.yahoo.labs.samoa.instances import Instances
import math
import random
import csv
import time

print('All imports OK!')

## 2. Helper Classes

In [ ]:
class newWindow():
    def __init__(self, schema, instances=None):
        self.window = Instances(schema.get_moa_header(), 0)
        self.schema = schema
        if instances is not None:
            for i in range(instances.numInstances()):
                self.window.add(instances.get(i))
    
    def add_instance(self, instance):
        self.window.add(instance.java_instance.getData())

    def add_window(self, nwindow):
        rw = nwindow.get_window()
        for i in range(nwindow.size()):
            self.window.add(rw.get(i))

    def remove_instance(self, position=0):
        self.window.delete(position)

    def get_instance(self, position=0):
        return self.window.get(position)

    def get_window(self):
        return self.window
    
    def get_schema(self):
        return self.schema
    
    def size(self):
        return self.window.numInstances()


class SearchNeighbours():
    def __init__(self, search_method):
        assert search_method in ['LinearNNSearch', 'KDTree']
        self.search_method_str = search_method
        if self.search_method_str == 'LinearNNSearch':
            self.search_method = LinearNNSearch()
        else:
            self.search_method = KDTree()

    def do_search(self, window, instance, arg_k):
        self.search_method.setInstances(window.get_window())
        k = min(arg_k, window.size())
        neighbors = self.search_method.kNearestNeighbours(instance.java_instance.getData(), k)
        return newWindow(window.get_schema(), neighbors)

print('Helpers loaded!')

## 3. KNN Classifiers

In [ ]:
# ==================== KNNBase (provided) ====================
class KNNBase(Classifier):
    def __init__(self, schema, k=3, window_size=1000, search_method='LinearNNSearch'):
        super().__init__(schema)
        self.k = k
        self.window_size = window_size
        self.search = SearchNeighbours(search_method)
        self.window = newWindow(schema)

    def __str__(self):
        return 'Base kNN'

    def train(self, instance):
        if self.window.size() >= self.window_size:
            self.window.remove_instance()
        self.window.add_instance(instance)
    
    def predict(self, instance):
        try:
            if self.window and self.window.size() > 0:
                neighbors = self.search.do_search(self.window, instance, self.k)
                votes = [0] * (self.schema.get_num_classes())
                for i in range(neighbors.size()):
                    class_idx = int(neighbors.get_instance(i).classValue())
                    votes[class_idx] += 1
                return votes.index(max(votes))
        except Exception as e:
            print('Error: kNN search failed.', e)
            return 0

    def predict_proba(self, instance):
        pass


# ==================== KNN-CW (Per Class Window) ====================
class KNN_cw_1689664(Classifier):
    def __init__(self, schema, k=3, w=1000, search_method='LinearNNSearch'):
        super().__init__(schema)
        self.k = k
        self.w = w
        self.search = SearchNeighbours(search_method)
        self.class_window = {}  # label -> newWindow

    def __str__(self):
        return 'KNN-cw'

    def train(self, instance):
        label = int(instance.java_instance.getData().classValue())
        if label not in self.class_window:
            self.class_window[label] = newWindow(self.schema)
        self.class_window[label].add_instance(instance)
        if self.class_window[label].size() > self.w:
            self.class_window[label].remove_instance(0)

    def predict(self, instance):
        try:
            # Stage 1: k nearest from each class window
            all_candidates = newWindow(self.schema)
            for label, cw in self.class_window.items():
                if cw.size() > 0:
                    local_k = min(self.k, cw.size())
                    neighbors = self.search.do_search(cw, instance, local_k)
                    all_candidates.add_window(neighbors)
            if all_candidates.size() == 0:
                return 0
            # Stage 2: global k nearest from all candidates
            final_k = min(self.k, all_candidates.size())
            final_neighbors = self.search.do_search(all_candidates, instance, final_k)
            # Stage 3: majority vote
            votes = [0] * self.schema.get_num_classes()
            for i in range(final_neighbors.size()):
                class_idx = int(final_neighbors.get_instance(i).classValue())
                votes[class_idx] += 1
            return votes.index(max(votes))
        except Exception as e:
            print('Error: kNN-cw search failed.', e)
            return 0

    def predict_proba(self, instance):
        pass


# ==================== KNN-ASW (Age-Weighted Sampling) ====================
LOG_1_2 = math.log(1.2)

class KNN_asw_1689664(Classifier):
    def __init__(self, schema, k=3, w=1000, search_method='LinearNNSearch'):
        super().__init__(schema)
        self.k = k
        self.w = w
        self.search = SearchNeighbours(search_method)
        self.window = newWindow(schema)
        self.ages = []
        random.seed(1689664)

    def __str__(self):
        return 'KNN-asw'

    def train(self, instance):
        # Increment age of all existing instances
        self.ages = [a + 1 for a in self.ages]
        # Add new instance with age = 1
        self.window.add_instance(instance)
        self.ages.append(1)
        # Eject one if window exceeds w
        if self.window.size() > self.w:
            self._eject_one()

    def _eject_one(self):
        n = self.window.size()
        # Log-sum-exp trick to avoid overflow
        log_weights = [self.ages[i] * LOG_1_2 for i in range(n)]
        max_log = max(log_weights)
        weights = [math.exp(lw - max_log) for lw in log_weights]
        total = sum(weights)
        # Sample one index to eject
        r = random.random() * total
        cumulative = 0.0
        eject_idx = n - 1
        for i in range(n):
            cumulative += weights[i]
            if cumulative >= r:
                eject_idx = i
                break
        self.window.remove_instance(eject_idx)
        self.ages.pop(eject_idx)

    def predict(self, instance):
        try:
            if self.window and self.window.size() > 0:
                neighbors = self.search.do_search(self.window, instance, self.k)
                votes = [0] * self.schema.get_num_classes()
                for i in range(neighbors.size()):
                    class_idx = int(neighbors.get_instance(i).classValue())
                    votes[class_idx] += 1
                return votes.index(max(votes))
        except Exception as e:
            print('Error: kNN-asw search failed.', e)
            return 0

    def predict_proba(self, instance):
        pass

print('All classifiers loaded!')

## 4. Quick Sanity Check
Run a small test first to make sure everything works before running all 24 experiments.

In [ ]:
# Quick test with 1000 instances
elec = Electricity()
bknn = KNNBase(schema=elec.get_schema(), k=3, window_size=50)
result = prequential_evaluation(stream=elec, learner=bknn, max_instances=1000)
print(f'KNNBase sanity check: accuracy={result.cumulative.accuracy():.3f}')

elec2 = Electricity()
cw = KNN_cw_1689664(schema=elec2.get_schema(), k=3, w=50)
result2 = prequential_evaluation(stream=elec2, learner=cw, max_instances=1000)
print(f'KNN-cw sanity check: accuracy={result2.cumulative.accuracy():.3f}')

elec3 = Electricity()
asw = KNN_asw_1689664(schema=elec3.get_schema(), k=3, w=50)
result3 = prequential_evaluation(stream=elec3, learner=asw, max_instances=1000)
print(f'KNN-asw sanity check: accuracy={result3.cumulative.accuracy():.3f}')

print('\nAll sanity checks passed! Ready to run full experiments.')

## 5. Run All 24 Experiments

In [ ]:
STUDENT_ID = '1689664'
MAX_INSTANCES = 50000

CONFIGS = [
    {'k': 3,  'w': 500},
    {'k': 21, 'w': 500},
    {'k': 3,  'w': 100},
    {'k': 21, 'w': 100},
]

DATASETS = {
    'electricity': Electricity,
    'rbf': RBFm_100k,
}

ALGORITHMS = {
    'knn': {'class': KNNBase, 'window_param': 'window_size'},
    'knn-cw': {'class': KNN_cw_1689664, 'window_param': 'w'},
    'knn-asw': {'class': KNN_asw_1689664, 'window_param': 'w'},
}

results = []
total = len(ALGORITHMS) * len(DATASETS) * len(CONFIGS)
count = 0

for algo_name, algo_info in ALGORITHMS.items():
    for dataset_name, dataset_class in DATASETS.items():
        for config in CONFIGS:
            count += 1
            k, w = config['k'], config['w']
            
            stream = dataset_class()
            schema = stream.get_schema()
            params = {'schema': schema, 'k': k, algo_info['window_param']: w}
            learner = algo_info['class'](**params)
            
            result = prequential_evaluation(
                stream=stream, learner=learner, max_instances=MAX_INSTANCES
            )
            
            accuracy = round(result.cumulative.accuracy(), 3)
            recall = round(result.cumulative.recall(), 3)
            precision = round(result.cumulative.precision(), 3)
            wallclock = round(result.wallclock(), 3)
            
            row = {
                'student_id': STUDENT_ID,
                'dataset': dataset_name,
                'knn_identifier': algo_name,
                'window_size': w,
                'k': k,
                'accuracy': accuracy,
                'recall': recall,
                'precision': precision,
                'wallclock': wallclock,
            }
            results.append(row)
            
            print(f'[{count}/{total}] {algo_name} | {dataset_name} | k={k}, w={w} | '
                  f'acc={accuracy}, rec={recall}, prec={precision}, time={wallclock}s')

print(f'\nDone! {len(results)} experiments completed.')

## 6. Generate CSV

In [ ]:
OUTPUT_FILE = '1689664_results.csv'

fieldnames = [
    'student_id', 'dataset', 'knn_identifier', 'window_size',
    'k', 'accuracy', 'recall', 'precision', 'wallclock'
]

with open(OUTPUT_FILE, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f'CSV written to {OUTPUT_FILE}')
print(f'Total rows: {len(results)}')

# Display results as a table
print('\n' + '='*100)
print(f'{"Algo":<10} {"Dataset":<15} {"k":<5} {"w":<6} {"Accuracy":<12} {"Recall":<10} {"Precision":<12} {"Time(s)":<10}')
print('='*100)
for r in results:
    print(f'{r["knn_identifier"]:<10} {r["dataset"]:<15} {r["k"]:<5} {r["window_size"]:<6} '
          f'{r["accuracy"]:<12} {r["recall"]:<10} {r["precision"]:<12} {r["wallclock"]:<10}')

## 7. Download CSV
If running on Colab, download the file:

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('Download started!')
except ImportError:
    print(f'Not on Colab. CSV saved locally as: {OUTPUT_FILE}')